# Data Cleaning

Let's go through all the columns and prepare the data to be used during analysis.

In [3]:
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [4]:
#warnings.filterwarnings("ignore")

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
plt.rcParams["figure.facecolor"] = "w"
pd.plotting.register_matplotlib_converters()
pd.set_option("display.float_format", lambda x: "%.3f" % x)

df_original = pd.read_csv("data/eda.csv")
df = df_original.copy()

Since using modulo operator on floats can be deceiving we use a different alghorithm to check if floats adhere to certain stepsizes:

`scaled = df[<column>] * 1/<stepsize>`

`do_not_adhere = ~np.isclose(scaled, scaled.round())`


In [ ]:
# Meaning of Icons:
# - All good: ✅ 
# - Correction helpful: ⚠️ 
# - Decision or Correction needed: ❌


# column names: 
# - lower case ✅
# - underscores✅


# id: 
# - integer ✅
# - no missing ✅


# bedrooms: 
# - float but only whole values ⚠️
# - non missing ✅

not_integer = ~np.isclose(df["bedrooms"], df["bedrooms"].round())
print(f"Bedrooms other than whole value: {not_integer.sum()}") # == 0

# - switch to integer
df["bedrooms"] = df["bedrooms"].round().astype(int)


# bathrooms: 
# - float and only quarter steps are allowed ✅
# - none missing ✅
scaled = df["bathrooms"] * 4
not_quarter_step = ~np.isclose(scaled, scaled.round())
print(f"Bathrooms other than quarter steps: {not_quarter_step.sum()}")


# sqft_living:
# - float but only whole values ⚠️
# - none missing ✅

# - switch to integer
df["sqft_living"] = df["sqft_living"].round().astype(int)

# sqft_lot:
# - float but only whole values ⚠️
# - none missing ✅

# - switch to integer
df["sqft_lot"] = df["sqft_lot"].round().astype(int)

In [ ]:
# floors:
# - float and non missing ✅
# - stepsize of 0.5 ✅
scaled = df["floors"] * 2
not_half_step = ~np.isclose(scaled, scaled.round())
print(f"Floors other than half steps: {not_half_step.sum()}")

In [ ]:
# waterfront
# - should be boolean or category (yes/no) ❌
# - missing values ❌
# Check different values
df["waterfront"].value_counts()

# Values 0 or 1 (and NaN)
# Since where a waterfront is available it is a major selling point
# we assume that only where there is no waterfront available one would exlude this info
# => NaN to 0
df["waterfront"] = df["waterfront"].fillna(0)

# the proportions haven't changed much also
print(f"Proportions before: {df_original['waterfront'].value_counts(normalize=True)}")
print(f"Proportions after: {df['waterfront'].value_counts(normalize=True)}")

#switch from numeric to boolean to be more precise
df["waterfront"] = df["waterfront"].astype(bool)
print(f"Proportions after type change: {df['waterfront'].value_counts(normalize=True)}")

In [9]:
# view
# - missing values ❌
# - float instead of category or integers ⚠️

# We keep the rows with the missing view for now since they are just a few
# But switch to nullable integer to be more precise
df["view"] = df["view"].astype("Int64")


In [ ]:
# condition
# - Values are integer and in correct range ✅
# - no missing values ✅

df.condition.value_counts()
df.condition.isna().sum()

In [ ]:
# grade
# - Values are integer and in correct range ✅
# - no missing values ✅

df.grade.value_counts()
df.grade.isna().sum()

In [ ]:
# sqft_above
# - float but only whole values ⚠️
# - none missing ✅
df.sqft_above.isna().sum()

# switch to integer
df["sqft_above"] = df["sqft_above"].round().astype(int)

In [ ]:
# sqft_basement
# - float but only whole values ⚠️
# - missing values ❌

# Validate assumption: sqft_living = sqft_above  + sqft_basement

# Get needed data where sqft_basement is not Nan. 
# The others we know already are complete
sqrf_data = df[["sqft_living", "sqft_above", "sqft_basement"]][~df["sqft_basement"].isna()]

# From that, get only the rows that follow our assumption
sqrf_data_valid = sqrf_data.query("sqft_above + sqft_basement == sqft_living")

# Check our assumption
print(f"Our assumtpion is {sqrf_data.equals(sqrf_data_valid)}") # == True

#check all whole values
not_integer = ~np.isclose(sqrf_data["sqft_basement"], sqrf_data["sqft_basement"].round())
print(f"sqrf_data other than whole value: {not_integer.sum()}") # == 0

In [14]:
# sqft_basement
# Replace NaN with "sqft_living - sqft_above"
# Easy way, do it for every row, even where sqft_basement was already set
# We proved already that this will not change any existing data
df["sqft_basement"] = df["sqft_living"] - df["sqft_above"]

# dtype will also be int64 now since we set sqft_living and sqft_above to int64 earlier

In [ ]:
# yr_built
# - integer and in a valid range ✅
# - no values missing ✅
print(df["yr_built"].min())
print(df["yr_built"].max())
print(df["yr_built"].info())

In [ ]:
# yr_renovated
# - year is float and order of magnitude too high ❌
# - 3848 missing values ❌
# - 0 as year ❌

print(f"Number of missing values: {df["yr_renovated"].isna().sum()}")
print(f"Earliest renovation: {df.query("yr_renovated > 0")["yr_renovated"].min()}")
print(f"Latest renovation: {df["yr_renovated"].max()}")

# Check if we can conclude anything from houses with a 0 and houses with a NaN
# Hypothesis: If only houses with recent build date have those values it means they were not renovated
yr_renovated_na = df[df["yr_renovated"].isna()]
print(f"Oldest house with NaN: {yr_renovated_na["yr_built"].min()}")
print(f"Newest house with NaN: {yr_renovated_na["yr_built"].max()}")

yr_renovated_0 = df[df["yr_renovated"] == 0]
print(f"Oldest house with 0: {yr_renovated_0["yr_built"].min()}")
print(f"Newest house with 0: {yr_renovated_0["yr_built"].max()}")

fig, axes = plt.subplots(2, 1)
sns.histplot(
        data=yr_renovated_0,
        x="yr_built",
        alpha=0.5,
        ax=axes[0],
        bins=11
    )
axes[0].set_title("Houses built with a 0", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Year")

sns.histplot(
        data=yr_renovated_na,
        x="yr_built",
        alpha=0.5,
        ax=axes[1],
        bins=11
    )
axes[1].set_title("House built with NaN", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Year")
    
plt.tight_layout()
plt.show()


In [ ]:
# yr_renovated
# Conclusion:
# Also houses with pretty old build date have both 0s and NaNs.
# It is unlikely the values mean "not renovated at all".
# We should just read them as "no information" and use NaN exclusively
df.loc[df["yr_renovated"] == 0, "yr_renovated"] = np.nan
df["yr_renovated"].info()

# Adjust values and type
df["yr_renovated"] = (df["yr_renovated"] / 10).round().astype("Int64")
df["yr_renovated"].head()

In [ ]:
#zipcode
# - stored as integer but is a nominal identifier, not a quantity ⚠️
# - zip code range is ok ✅

# switch to category to avoid it being treated as an ordinal/numeric quantity
df["zipcode"] = df["zipcode"].astype("category")

#King County, Washington uses ZIP codes primarily starting with the 980xx and 981xx ranges
print(f"Lowest code: {df["zipcode"].cat.categories.min()}")
print(f"Highest code: {df["zipcode"].cat.categories.max()}")



In [ ]:
# latitude
# - In correct range (~ 47.1 - 47.8) ✅
# - float ✅
# - no values missing ✅
df["lat"].info()
df["lat"].describe()



In [ ]:
#longitude
# - In correct range (~ -122,5 -121,1) ✅
# - float ✅
# - no values missing ✅

df["long"].info()
df["long"].describe()

In [ ]:
#sqft_living15
# - float ⚠️
# - no missing values ✅
# - in good range (not higher than available in house data) ✅

# whole values?
not_integer = ~np.isclose(df["sqft_living15"], df["sqft_living15"].round())
print(f"sqft_living15 other than whole value: {not_integer.sum()}")
# type as int
df["sqft_living15"] = df["sqft_living15"].round().astype(int)

# Are there higher squarefeet mentioned than possible?
higher_sqft = df.loc[df["sqft_living15"] > df["sqft_living"].max(), "sqft_living15"].count()
print(f"Amount of sqrft mentioned that are higher than available: {higher_sqft}")

In [ ]:
#sqft_lot15
# - float ⚠️
# - no missing values ✅
# - in good range (not higher than available in house data) ✅

df["sqft_lot15"].info()

# whole values?
not_integer = ~np.isclose(df["sqft_lot15"], df["sqft_lot15"].round())
print(f"sqft_lot15 other than whole value: {not_integer.sum()}")
# type as int
df["sqft_lot15"] = df["sqft_lot15"].round().astype(int)

# Are there higher squarefeet mentioned than possible?
higher_sqft = df.loc[df["sqft_lot15"] > df["sqft_lot"].max(), "sqft_lot15"].count()
print(f"Amount of sqrft mentioned that are higher than available: {higher_sqft}")

In [ ]:
#selling_price
# - float ✅
# - no missing values ✅
# - range reasonable(78k - 7.7m) ✅

df["selling_price"].info()
df["selling_price"].describe()

In [ ]:
#selling_date
# - no missing values ✅
# - string instead of date ❌
# - range ok

df["selling_date"].info()
df["selling_date"].describe()

#simple check that format is consistent (only works on strings the on first run)
#print(f"Number of inconsistent dates: {df["selling_date"].loc[df["selling_date"].str.fullmatch(r"^\d\d\d\d-\d\d-\d\d$") == False].count()}")

# change type to datetime
df["selling_date"] = pd.to_datetime(df["selling_date"])

df["selling_date"].describe()

In [ ]:
# write out clean data
df.to_csv("data/eda_clean.csv", index=False)